## Imports

## Submission

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Отключаем ворнинги wandb (чтобы они не спамили в аутпут на Kaggle)
os.environ["WANDB_DISABLED"] = "true"

# ==========================================
# 1. ПАРАМЕТРЫ
# ==========================================
TRAIN_PATH = '/kaggle/input/competitions/lost-interpreter-aicc-round-5/train.csv'
TEST_PATH = '/kaggle/input/competitions/lost-interpreter-aicc-round-5/test.csv'
MODEL_NAME = 'BAAI/bge-small-en-v1.5'  # Быстрая и качественная модель
MAX_LENGTH = 128                       # Максимальная длина токенов
EPOCHS = 3
BATCH_SIZE = 16

# ==========================================
# 2. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ==========================================
print("⏳ Загрузка данных...")
df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Для регрессии (MSELoss) HuggingFace требует колонку 'label' в формате float32
df['label'] = df['output'].astype(np.float32)
df['program'] = df['program'].astype(str)
test_df['program'] = test_df['program'].astype(str)

# Разбиваем на train / validation (90% / 10%)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Переводим в формат Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# ==========================================
# 3. ТОКЕНИЗАЦИЯ
# ==========================================
print("⏳ Токенизация датасета...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["program"], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Collator динамически добивает нулями тексты до максимальной длины В БАТЧЕ (ускоряет обучение)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==========================================
# 4. МЕТРИКА И МОДЕЛЬ
# ==========================================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.squeeze() 
    mae = mean_absolute_error(labels, preds)
    return {"mae": mae}

# num_labels=1 переключает модель в режим регрессии
print(f"⏳ Загрузка модели {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

# ==========================================
# 5. НАСТРОЙКИ ОБУЧЕНИЯ (Trainer)
# ==========================================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2, # При валидации памяти тратится меньше, батч можно увеличить
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    fp16=torch.cuda.is_available(), # Ускорение на видеокартах (Tesla T4)
    report_to="none"
)

# ИСПРАВЛЕНИЕ: Убрали аргумент tokenizer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ==========================================
# 6. ОБУЧЕНИЕ
# ==========================================
print("🚀 Начинаем обучение...")
trainer.train()

# ==========================================
# 7. ПРЕДСКАЗАНИЕ И СОХРАНЕНИЕ
# ==========================================
print("📊 Делаем предсказания на тестовой выборке...")
test_results = trainer.predict(tokenized_test)

# Достаем вероятности и "сплющиваем" массив
test_preds = test_results.predictions.squeeze()

# Округляем до ближайшего целого
test_pred_rounded = np.rint(test_preds).astype(int)

# Формируем итоговый файл
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "output": test_pred_rounded
})

submission.to_csv("submission.csv", index=False)
print("✅ Успешно! Файл 'submission.csv' сохранен.")
display(submission.head())

⏳ Загрузка данных...
⏳ Токенизация датасета...


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

📊 Делаем предсказания на тестовой выборке...


✅ Успешно! Файл 'submission.csv' сохранен.


,ID,output
0,0,-1
1,1,-1
2,2,-1
3,3,-1
4,4,0


## Evaluation